# Official vCache on the SemanticCacheController workload


In [ ]:
!git clone https://github.com/jg-eno/SemanticCacheController
%cd SemanticCacheController

Cloning into 'SemanticCacheController'...
remote: Enumerating objects: 226, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 226 (delta 101), reused 186 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (226/226), 1.78 MiB | 5.40 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/content/SemanticCacheController


In [ ]:
# 1) Where is your repo? Edit this path if yours differs.
import os
REPO = "/content/SemanticCacheController"
assert os.path.exists(f"{REPO}/src/baseline_models/vcache_official.py"), "vcache_official.py not found - fix REPO or add the file"
os.environ["REPO"] = REPO
os.environ["VENV"] = "/content/venv311"
os.environ["VCACHE_DIR"] = "/content/vCache"
print("ok")

ok


In [ ]:
%%bash
# 2) Python 3.11 venv (uv downloads 3.11 for us)
set -e
pip -q install uv
uv venv --python 3.11 "$VENV"
"$VENV/bin/python" --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 93.5 MB/s eta 0:00:00
Python 3.11.17


Using CPython 3.11.17
Creating virtual environment at: /content/venv311
Activate with: source /content/venv311/bin/activate


In [ ]:
%%bash
# 3) Official vCache at the pinned commit + this repo's dependencies (big install, several minutes)
set -e
[ -d "$VCACHE_DIR" ] || git clone https://github.com/vcache-project/vCache "$VCACHE_DIR"
git -C "$VCACHE_DIR" checkout 50694eebf0c870231030f73707cb6543d47f0411
uv pip install --python "$VENV/bin/python" -e "$VCACHE_DIR" scikit-learn gymnasium sentence-transformers
# vCache pins numpy 1.26, so use pyarrow/datasets versions that still support numpy 1.x
uv pip install --python "$VENV/bin/python" "pyarrow<19" "datasets<4"
"$VENV/bin/python" -c "import numpy, pyarrow, datasets, sentence_transformers; print('numpy', numpy.__version__, '| pyarrow', pyarrow.__version__, '| sentence_transformers OK')"
"$VENV/bin/python" -c "import vcache; print('vcache imports OK')"

numpy 1.26.4 | pyarrow 18.1.0 | sentence_transformers OK
INFO 10-19 19:46:35 [__init__.py:216] Automatically detected platform cuda.
vcache imports OK


Cloning into '/content/vCache'...
Note: switching to '50694eebf0c870231030f73707cb6543d47f0411'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 50694ee Add resource/throughput benchmark metrics and offline CI smoke coverage (#99)
Using Python 3.11.17 environment at: /content/venv311
Resolved 230 packages in 4.97s
Prepared 229 packages in 1m 46s
Installed 230 packages in 1.31s
 + accelerate==1.15.0
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiohttp-retry==2.9.1
 + aiosignal==1.4.0

In [ ]:
%%bash
# 4) Adapter unit tests (stand-ins, no official policy) - should all pass
cd "$REPO"
"$VENV/bin/python" -m unittest src.tests.test_vcache_official_adapter src.tests.test_vcache_baseline -v 2>&1 | tail -25

test_missing_package_gives_install_hint (src.tests.test_vcache_official_adapter.ImportTests.test_missing_package_gives_install_hint) ... skipped 'official vCache is installed'
test_counts_statuses_and_metrics (src.tests.test_vcache_official_adapter.RunWorkloadTests.test_counts_statuses_and_metrics) ... ok
test_empty_workload_is_rejected (src.tests.test_vcache_official_adapter.RunWorkloadTests.test_empty_workload_is_rejected) ... ok
test_same_intent_gets_same_integer_id (src.tests.test_vcache_official_adapter.RunWorkloadTests.test_same_intent_gets_same_integer_id) ... ok
test_unlabeled_rows_are_unevaluable (src.tests.test_vcache_official_adapter.RunWorkloadTests.test_unlabeled_rows_are_unevaluable) ... ok
test_times_out_when_update_never_arrives (src.tests.test_vcache_official_adapter.WaitingTests.test_times_out_when_update_never_arrives) ... ok
test_waits_for_background_update_to_finish (src.tests.test_vcache_official_adapter.WaitingTests.test_waits_for_background_update_to_finish) ...

In [ ]:
%%bash
# 5) Check every import the adapter needs before the real run
cd "$REPO"
"$VENV/bin/python" - <<'PY'
from vcache.config import VCacheConfig
from vcache.inference_engine.inference_engine import InferenceEngine
from vcache.main import VCache
from vcache.vcache_core.cache.embedding_engine.embedding_engine import EmbeddingEngine
from vcache.vcache_core.cache.embedding_store.embedding_metadata_storage.strategies.in_memory import InMemoryEmbeddingMetadataStorage
from vcache.vcache_core.cache.embedding_store.vector_db.strategies.hnsw_lib import HNSWLibVectorDB
from vcache.vcache_core.cache.eviction_policy.strategies.no_eviction import NoEvictionPolicy
from vcache.vcache_core.similarity_evaluator.similarity_evaluator import SimilarityEvaluator
from vcache.vcache_policy.strategies.verified import VerifiedDecisionPolicy
print("all official imports OK")
PY

INFO 10-19 19:46:51 [__init__.py:216] Automatically detected platform cuda.
all official imports OK


In [ ]:
%%bash
# 6) Run the official policy on the example workload (it mostly explores on 96 requests - expected)
cd "$REPO"
"$VENV/bin/python" -m src.baseline_models.vcache_official \
  --workload src/datasets/example_workload.jsonl --deltas 0.01 0.02 0.05

INFO 10-19 19:47:04 [__init__.py:216] Automatically detected platform cuda.
{
  "seed": 0,
  "settings": {
    "workload": "src/datasets/example_workload.jsonl",
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "model_revision": null
  },
  "results": {
    "official_vCache_delta_0.01": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.4583333333333333,
      "mean_environment_reward": 0.4499999999999999,
      "explorations": 43
    },
    "official_vCache_delta_0.02": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.25,
      "mean_environment_reward": 0.6999999999999998,
      "explorations": 23
    },
    "official_vCache_delta_0.05": {
      "requests": 96,
      "evaluable_labeled_requests": 96,
      "false_hit_rate": 0.0,
      "recomputation_rate": 0.16666666666666666,
      "mean_environment_reward": 0.7999999999999998,
 

/content/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/content/venv311/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/content/venv311/lib/python3.11/site-packages/sklearn/